In [91]:
import sys
import importlib
sys.path.append('../')  # Adjust the path as needed

import utilities.functions as functions
import utilities.plot as plot

# Reload the module to reflect the changes
importlib.reload(functions)
importlib.reload(plot)

<module 'utilities.plot' from '/Users/xuechenkan/potts_model_test/ms1/../utilities/plot.py'>

In [92]:
IN_seq_path = 'IN/data/in.reduce4.seq'
PR_seq_path = 'PR/data/pr.exper.reduce4.seq'    
RT_seq_path = 'RT/data/rt.reduce4.seq'

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_consensus = 'IN/data/in.consensus.reduce4.seq'
with open(IN_consensus, 'r') as f:
    IN_consensus_seq = f.read().strip()
# print("IN consensus sequence:", IN_consensus_seq)
PR_consensus = 'PR/data/pr.consensus.reduce4.seq'
with open(PR_consensus, 'r') as f:
    PR_consensus_seq = f.read().strip()
RT_consensus = 'RT/data/rt.consensus.reduce4.seq'
with open(RT_consensus, 'r') as f:
    RT_consensus_seq = f.read().strip()

# print(IN_consensus_seq)
    
IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux',1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux',0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux',0)

IN_J = functions.load_J_dict('IN/data/J.npy',1,263)
PR_J = functions.load_J_dict('PR/data/J_PR.npy',1,99)
RT_J = functions.load_J_dict('RT/data/J_RT.npy',39,226)

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')
RT_all_seq_unreduced = functions.read_seq('RT/data/rt.fullseq')

In [93]:
def count_gof_total_and_withDMC(seqs, redux, mut_pair, J, min_pos, max_pos):
    pair1,pair2 = functions.split_pairs(mut_pair)
    reduced_pair1 = functions.unreduced_to_reduced(redux, pair1)
    reduced_pair2 = functions.unreduced_to_reduced(redux, pair2)
    # print(f"Reduced pair 1: {reduced_pair1}")
    # print(f"Reduced pair 2: {reduced_pair2}")
    wt1, pos1, mt1 = functions.split_pair(reduced_pair1)
    wt2, pos2, mt2 = functions.split_pair(reduced_pair2)

    count_withDMC = 0
    count_total = 0
    for seq in seqs:
        # de1 = functions.calculate_delta_e(reduced_pair1, seq, J, min_pos, max_pos)
        # de2 = functions.calculate_delta_e(reduced_pair2, seq, J, min_pos, max_pos)
        # de12 = functions.calculate_delta_e_double(reduced_pair1, reduced_pair2, seq, J, min_pos, max_pos)

        de1, de2, de12, dde12 = functions.calculate_dde_v3(reduced_pair1, reduced_pair2, seq, J, min_pos, max_pos)

        if de12 > de1 and de12 > de2 and de12 > 0:
            # print(reduced_pair1, reduced_pair2, de12, de1, de2, de12-de1-de2)
            count_total += 1
            if seq[pos1-min_pos] == mt1 and seq[pos2-min_pos] == mt2:
                count_withDMC += 1
    print(f"Total sequences: {len(seqs)}")
    print(f"Total gof: {count_total}")    
    print(f"Total gof with DMC: {count_withDMC}")
    return count_withDMC, count_total

In [94]:
# PR_weights_path = 'PR/data/pr.exper.weights.txt'
# len_PR_all_seqs = len(PR_all_seq)
# with open(PR_weights_path, 'r') as f:
#     PR_weights = [float(line.strip()) for line in f]

# # Ensure the weights list matches the IN_all_seq list
# assert len(PR_weights) == len_PR_all_seqs, "Weights and sequences must have the same length."

# PR_pairs = [
#     'D30N-N88D',
#     'V32I-I47V',
#     'G48V-I54A',
#     'D30N-K45Q',
#     'I54A-V82A',
#     'I54V-V82A',
#     'M46I-L76V',
#     'I54V-V82T',
#     'I54A-V82T',
#     'G48V-V82A',
#     'I54A-A71I',
#     'M46I-N88T',
#     'L90M-C95F',
#     'V32I-M46I',
#     'G48V-V82T',
#     'I54V-T91S',
#     'M46I-F53Y',
#     'M46L-K55R',
#     'M46L-V82A',
#     'M46I-K55R',
# ]
# for pair in PR_pairs:
#     print(f"Analyzing pair: {pair}")
#     count_withDMC, count_total = count_gof_total_and_withDMC(PR_all_seq, PR_redux, pair, PR_J, 1, 99)

In [95]:
import csv

PR_pairs = ['D30N-N88D']
pair = PR_pairs[0]
pair1, pair2 = functions.split_pairs(pair)
reduced_pair1 = functions.unreduced_to_reduced(PR_redux, pair1)
reduced_pair2 = functions.unreduced_to_reduced(PR_redux, pair2)

rows = []
for seq in PR_all_seq:
    de1, de2, de12, dde = functions.calculate_dde_v2(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    # de1 = functions.calculate_delta_e(reduced_pair1, seq, PR_J, 1, 99)
    # de2 = functions.calculate_delta_e(reduced_pair2, seq, PR_J, 1, 99)
    # de12 = functions.calculate_delta_e_double(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    rows.append({'de1': de1, 'de2': de2, 'de12': de12, 'dde12': dde})

output_csv = f'PR/data/{pair}_des.csv'
with open(output_csv, 'w', newline='') as csvfile:
    fieldnames = ['de1', 'de2', 'de12', 'dde12']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Output written to {output_csv}")

Output written to PR/data/D30N-N88D_des.csv


In [96]:
def read_float_file(path):
    with open(path, "r") as fh:
        return [float(line.strip()) for line in fh if line.strip()]

de1_file = read_float_file("PR/data/de.try.D30N")
de2_file = read_float_file("PR/data/de.try.N88D")
de12_file = read_float_file("PR/data/de.D30N-N88D")

n_rows = len(rows)
if not (len(de1_file) == len(de2_file) == len(de12_file) == n_rows):
    print(
        f"Length mismatch -> rows: {n_rows}, "
        f"de1_file: {len(de1_file)}, de2_file: {len(de2_file)}, de12_file: {len(de12_file)}"
    )
else:
    threshold = 0.001
    mismatch_count = 0

    for i, (r, d1f, d2f, d12f) in enumerate(zip(rows, de1_file, de2_file, de12_file)):
        d1 = abs(float(r["de1"]) - d1f)
        d2 = abs(float(r["de2"]) - d2f)
        d12 = abs(float(r["de12"]) - d12f)
        ddef = d12f - d1f - d2f
        ddde = abs(float(r["dde12"]) - ddef)

        if d1 > threshold or d2 > threshold or d12 > threshold or ddde > threshold:
            mismatch_count += 1
            print(
                f"idx={i} | "
                f"de1: mine={float(r['de1']):.6f}, file={d1f:.6f}, diff={d1:.6f} | "
                f"de2: mine={float(r['de2']):.6f}, file={d2f:.6f}, diff={d2:.6f} | "
                f"de12: mine={float(r['de12']):.6f}, file={d12f:.6f}, diff={d12:.6f} | "
                f"dde: mine={float(r['dde12']):.6f}, file={ddef:.6f}, diff={ddde:.6f}"
            )

    print(f"Total mismatches (> {threshold}): {mismatch_count}")

idx=10 | de1: mine=-3.873347, file=-2.510072, diff=1.363275 | de2: mine=-4.578923, file=-4.578924, diff=0.000001 | de12: mine=-3.841395, file=-2.478118, diff=1.363278 | dde: mine=4.610875, file=4.610879, diff=0.000003
idx=11 | de1: mine=-3.793690, file=-2.430414, diff=1.363276 | de2: mine=-4.065769, file=-4.065770, diff=0.000001 | de12: mine=-3.248585, file=-1.885305, diff=1.363279 | dde: mine=4.610874, file=4.610879, diff=0.000004
idx=78 | de1: mine=-0.898351, file=3.712521, diff=4.610872 | de2: mine=-3.339725, file=1.271153, diff=4.610877 | de12: mine=0.372797, file=9.594553, diff=9.221756 | dde: mine=4.610872, file=4.610879, diff=0.000006
idx=199 | de1: mine=-2.585642, file=-2.585647, diff=0.000005 | de2: mine=-4.465681, file=0.145197, diff=4.610878 | de12: mine=-2.440449, file=2.170428, diff=4.610877 | dde: mine=4.610874, file=4.610879, diff=0.000004
idx=201 | de1: mine=-2.669271, file=-2.669276, diff=0.000006 | de2: mine=-4.563401, file=0.047473, diff=4.610874 | de12: mine=-2.6218

In [97]:
# Recompute mismatch indexes and calculate dde from file values:
# dde_file = de12_file - de1_file - de2_file

mismatch_dde_from_file = []

for i, (r, d1f, d2f, d12f) in enumerate(zip(rows, de1_file, de2_file, de12_file)):
    d1_diff = abs(float(r["de1"]) - d1f)
    d2_diff = abs(float(r["de2"]) - d2f)
    d12_diff = abs(float(r["de12"]) - d12f)

    if d1_diff > threshold or d2_diff > threshold or d12_diff > threshold:
        dde_file = d12f - d1f - d2f
        mismatch_dde_from_file.append({"idx": i, "dde_file": dde_file})
        print(f"idx={i}, dde_file={dde_file:.6f}")

print(f"Total mismatch indexes: {len(mismatch_dde_from_file)}")

idx=10, dde_file=4.610879
idx=11, dde_file=4.610879
idx=78, dde_file=4.610879
idx=199, dde_file=4.610879
idx=201, dde_file=4.610879
idx=203, dde_file=4.610879
idx=208, dde_file=4.610879
idx=209, dde_file=4.610879
idx=210, dde_file=4.610879
idx=211, dde_file=4.610879
idx=213, dde_file=4.610879
idx=214, dde_file=4.610879
idx=215, dde_file=4.610879
idx=217, dde_file=4.610879
idx=219, dde_file=4.610879
idx=221, dde_file=4.610879
idx=222, dde_file=4.610879
idx=224, dde_file=4.610879
idx=225, dde_file=4.610879
idx=227, dde_file=4.610879
idx=229, dde_file=4.610879
idx=230, dde_file=4.610879
idx=236, dde_file=4.610879
idx=237, dde_file=4.610879
idx=238, dde_file=4.610879
idx=239, dde_file=4.610879
idx=240, dde_file=4.610879
idx=242, dde_file=4.610879
idx=244, dde_file=4.610879
idx=246, dde_file=4.610879
idx=252, dde_file=4.610879
idx=276, dde_file=4.610879
idx=281, dde_file=4.610879
idx=286, dde_file=4.610879
idx=289, dde_file=4.610879
idx=291, dde_file=4.610879
idx=292, dde_file=4.610879
idx=

In [98]:
# Classify mismatch indexes at the two mutation sites for pair1/pair2 (e.g., D30N and N88D)

wt1, pos1, mt1 = functions.split_pair(pair1)  # pair1 already exists: 'D30N'
wt2, pos2, mt2 = functions.split_pair(pair2)  # pair2 already exists: 'N88D'

def site_state(residue, wt, mt):
    if residue == wt:
        return "wild type"
    if residue == mt:
        return "mutant"
    return "neither"

def overall_state(s1, s2):
    if s1 == "wild type" and s2 == "wild type":
        return "wild type"
    if s1 == "mutant" and s2 == "mutant":
        return "mutant"
    return "neither"

results = []
for item in mismatch_dde_from_file:   # list of {"idx": ..., "dde_file": ...}
    idx = item["idx"]
    seq = PR_all_seq_unreduced[idx]   # use unreduced amino-acid sequence

    res1 = seq[pos1 - 1]
    res2 = seq[pos2 - 1]

    s1 = site_state(res1, wt1, mt1)
    s2 = site_state(res2, wt2, mt2)

    results.append({
        "idx": idx,
        "pos1": f"{wt1}{pos1}{mt1}",
        "aa1": res1,
        "state1": s1,
        "pos2": f"{wt2}{pos2}{mt2}",
        "aa2": res2,
        "state2": s2,
        "overall": overall_state(s1, s2),
    })

# Print per-index classification
for r in results:
    print(
        f"idx={r['idx']}: "
        f"{r['pos1']} -> {r['aa1']} ({r['state1']}), "
        f"{r['pos2']} -> {r['aa2']} ({r['state2']}) | overall={r['overall']}"
    )

# Optional summary counts
summary = {"wild type": 0, "mutant": 0, "neither": 0}
for r in results:
    summary[r["overall"]] += 1

print("\nSummary (overall across both sites):", summary)

idx=10: D30N -> D (wild type), N88D -> T (neither) | overall=neither
idx=11: D30N -> D (wild type), N88D -> T (neither) | overall=neither
idx=78: D30N -> N (mutant), N88D -> D (mutant) | overall=mutant
idx=199: D30N -> N (mutant), N88D -> N (wild type) | overall=neither
idx=201: D30N -> N (mutant), N88D -> N (wild type) | overall=neither
idx=203: D30N -> N (mutant), N88D -> N (wild type) | overall=neither
idx=208: D30N -> D (wild type), N88D -> S (neither) | overall=neither
idx=209: D30N -> N (mutant), N88D -> D (mutant) | overall=mutant
idx=210: D30N -> N (mutant), N88D -> N (wild type) | overall=neither
idx=211: D30N -> N (mutant), N88D -> D (mutant) | overall=mutant
idx=213: D30N -> N (mutant), N88D -> N (wild type) | overall=neither
idx=214: D30N -> N (mutant), N88D -> - (neither) | overall=neither
idx=215: D30N -> N (mutant), N88D -> N (wild type) | overall=neither
idx=217: D30N -> N (mutant), N88D -> N (wild type) | overall=neither
idx=219: D30N -> N (mutant), N88D -> N (wild typ

In [99]:
import csv

def output_probs(prefix, min_pos, max_pos, all_seq, consensus_seq, redux, pairs, weights_path, J, output_csv):
    """
    Process epistasis data for a given prefix (e.g., IN, PR, RT).

    Parameters:
        prefix (str): Prefix for the dataset (e.g., 'IN', 'PR', 'RT').
        all_seq (list): List of all sequences.
        consensus_seq (str): Consensus sequence.
        redux (dict): Reduction dictionary.
        pairs (list): List of mutation pairs.
        weights_path (str): Path to the weights file.
        J (dict): Interaction matrix.
        output_csv (str): Output CSV file name.
    """
    len_all_seqs = len(all_seq)

    # Read weights from the file
    with open(weights_path, 'r') as f:
        weights = [float(line.strip()) for line in f]

    # Ensure the weights list matches the all_seq list
    assert len(weights) == len(all_seq), "Weights and sequences must have the same length."


    csv_total_data = []
    csv_data = []
    csv_weighted_data = []

    for pair in pairs:
        print(f"Processing pair: {pair}")
        pair1, pair2 = functions.split_pairs(pair)
        p1_reduced = functions.unreduced_to_reduced(redux, pair1)
        p2_reduced = functions.unreduced_to_reduced(redux, pair2)
        wt1, pos1, mt1 = functions.split_pair(p1_reduced)
        wt2, pos2, mt2 = functions.split_pair(p2_reduced)

        ########
        flip_counts = 0
        flip_without_DMC_count = 0
        flip_with_DMC_count = 0
        flip_with_one_mut_count = 0
        
        ########
        non_flip_counts = 0
        non_flip_without_DMC_count = 0
        non_flip_with_DMC_count = 0
        non_flip_with_one_mut_count = 0

        ########
        gof_counts = 0
        gof_without_DMC_count = 0
        gof_with_DMC_count = 0
        gof_with_one_mut_count = 0

        ########
        rescue_count = 0
        rescue_without_DMC_count = 0
        rescue_with_DMC_count = 0
        rescue_with_one_mut_count = 0

        ########
        compensatory_counts = 0
        compensatory_without_DMC_count = 0
        compensatory_with_DMC_count = 0
        compensatory_with_one_mut_count = 0

        ########
        noncompensatory_counts = 0
        noncompensatory_without_DMC_count = 0
        noncompensatory_with_DMC_count = 0
        noncompensatory_with_one_mut_count = 0

        flip_seqs = []
        flip_weights = []

        non_flip_seqs = []
        non_flip_weights = []

        gof_seqs = []
        gof_weights = []

        rescue_seqs = []
        rescue_weights = []

        comp_seqs = []
        comp_weights = []

        noncomp_seqs = []
        non_comp_weights = []
    ##############
        consensus_result = functions.calculate_dde_v3(p1_reduced, p2_reduced, consensus_seq, J, min_pos, max_pos)
        consensus_pair1_de, consensus_pair2_de, consensus_pair12_de, consensus_pair12_dde = consensus_result
        consensus_de_diff = consensus_pair1_de - consensus_pair2_de
    ##############
        total_counts = 0
        total_with_DMC_count = 0
        all_seqs = []

        curr_i = 0
        for seq in all_seq:

            # result_merged = functions.calculate_dde_v3(p1_reduced, p2_reduced, seq, J, min_pos, max_pos)

            # if result_merged is None:
            #     continue
            # pair1_de, pair2_de, pair12_de, pair12_dde = result_merged

            pair1_de = de1_file[curr_i]
            pair2_de = de2_file[curr_i]
            pair12_de = de12_file[curr_i]
            pair12_dde = pair12_de - pair1_de - pair2_de

            total_counts += 1
            all_seqs.append(seq)
            if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                total_with_DMC_count += 1
    ###############
            result_merged = functions.calculate_dde_v3(p1_reduced, p2_reduced, seq, J, min_pos, max_pos)
            if result_merged is None:
                continue
            pair1_de, pair2_de, pair12_de, pair12_dde = result_merged
    ###############
            de_diff_seq = pair1_de - pair2_de
    
    #flip/nonflip block TODO there are logical errors for one mutation count
            if consensus_de_diff * de_diff_seq < 0: #FLIPPED!!
                flip_counts += 1
                flip_seqs.append(seq)
                flip_weights.append(weights[curr_i])
                
                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    flip_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    flip_with_DMC_count += 1
                else:
                    flip_with_one_mut_count += 1

            elif consensus_de_diff * de_diff_seq > 0: #NON-FLIPPED
                non_flip_counts += 1
                non_flip_seqs.append(seq)
                non_flip_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    non_flip_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    non_flip_with_DMC_count += 1
                else:
                    non_flip_with_one_mut_count += 1
    ###############
            if pair1_de < pair12_de and pair2_de < pair12_de and pair12_de > 0:
                gof_counts += 1
                gof_seqs.append(seq)
                gof_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    gof_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    gof_with_DMC_count += 1
                else:
                    gof_with_one_mut_count += 1

            elif pair1_de < pair12_de and pair2_de < pair12_de:
                rescue_count += 1
                rescue_seqs.append(seq)
                rescue_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    rescue_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    rescue_with_DMC_count += 1
                else:
                    rescue_with_one_mut_count += 1

            elif pair1_de < pair12_de or pair2_de < pair12_de:
                compensatory_counts += 1
                comp_seqs.append(seq)
                comp_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    compensatory_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    compensatory_with_DMC_count += 1
                else:
                    compensatory_with_one_mut_count += 1
            else:
                noncompensatory_counts += 1
                noncomp_seqs.append(seq)
                non_comp_weights.append(weights[curr_i])

                if seq[pos1 - min_pos] == wt1 and seq[pos2 - min_pos] == wt2:
                    noncompensatory_without_DMC_count += 1
                elif seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2:
                    noncompensatory_with_DMC_count += 1
                else:
                    noncompensatory_with_one_mut_count += 1
            curr_i += 1
        p_SH_all = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in all_seqs]
        average_p_all = sum(p_SH_all) / len_all_seqs if p_SH_all else 0
        weighted_all_prob = sum(p * w for p, w in zip(p_SH_all, weights[:len_all_seqs])) / sum(weights[:len_all_seqs]) if p_SH_all else 0
        
        #total weights sum and weights sum for sequences with DMC
        weights_sum = sum(weights[:len_all_seqs])
        weights_with_DMC_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(all_seqs, weights[:len(all_seqs)]))

        # Calculate the actual_p values
        actual_p_all = total_with_DMC_count / total_counts if total_counts > 0 else 0
        actual_p_all_weighted = weights_with_DMC_sum / weights_sum if all_seqs else 0

        # Append data
        csv_total_data.append([pair,'Total', total_counts,total_with_DMC_count, average_p_all, actual_p_all, average_p_all, actual_p_all,weights_sum,weights_with_DMC_sum, weighted_all_prob, actual_p_all_weighted,weighted_all_prob, actual_p_all_weighted])
        # Calculate average probabilities
        p_SH_flip = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in flip_seqs]
        average_p_flip_subcategory = sum(p_SH_flip) / len(p_SH_flip) if p_SH_flip else 0
        average_p_flip_total = sum(p_SH_flip) / len_all_seqs if len_all_seqs else 0
        weighted_flip_prob_subcategory = sum(p * w for p, w in zip(p_SH_flip, flip_weights)) / sum(flip_weights) if flip_weights else 0
        weighted_flip_prob_total = sum(p * w for p, w in zip(p_SH_flip, flip_weights)) / sum(weights) if flip_weights else 0

        p_SH_nonflip = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in non_flip_seqs]
        average_p_nonflip_subcategory = sum(p_SH_nonflip) / len(p_SH_nonflip) if p_SH_nonflip else 0
        average_p_nonflip_total = sum(p_SH_nonflip) / len_all_seqs if len_all_seqs else 0
        weighted_nonflip_prob_subcategory = sum(p * w for p, w in zip(p_SH_nonflip, non_flip_weights)) / sum(non_flip_weights) if non_flip_weights else 0
        weighted_nonflip_prob_total = sum(p * w for p, w in zip(p_SH_nonflip, non_flip_weights)) / sum(weights) if non_flip_weights else 0 

        p_SH_gofs = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in gof_seqs]
        average_p_gof_subcategory = sum(p_SH_gofs) / len(p_SH_gofs) if p_SH_gofs else 0
        average_p_gof_total = sum(p_SH_gofs) / len_all_seqs if len_all_seqs else 0
        weighted_gof_prob_subcategory = sum(p * w for p, w in zip(p_SH_gofs, gof_weights)) / sum(gof_weights) if gof_weights else 0
        weighted_gof_prob_total = sum(p * w for p, w in zip(p_SH_gofs, gof_weights)) / sum(weights) if gof_weights else 0
        
        p_SH_rescues = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in rescue_seqs]
        average_p_rescue_subcategory = sum(p_SH_rescues) / len(p_SH_rescues) if p_SH_rescues else 0
        average_p_rescue_total = sum(p_SH_rescues) / len_all_seqs if len_all_seqs else 0
        weighted_rescue_prob_subcategory = sum(p * w for p, w in zip(p_SH_rescues, rescue_weights)) / sum(rescue_weights) if rescue_weights else 0
        weighted_rescue_prob_total = sum(p * w for p, w in zip(p_SH_rescues, rescue_weights)) / sum(weights) if rescue_weights else 0

        # average_p_gof_rescue = (sum(p_SH_gofs) + sum(p_SH_rescues)) / (len(p_SH_gofs) + len(p_SH_rescues)) if (len(p_SH_gofs) + len(p_SH_rescues)) > 0 else 0
        p_SH_comps = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in comp_seqs]
        average_p_compensatory_subcategory = sum(p_SH_comps) / len(p_SH_comps) if p_SH_comps else 0
        average_p_compensatory_total = sum(p_SH_comps) / len_all_seqs if len_all_seqs else 0
        weighted_comp_prob_subcategory = sum(p * w for p, w in zip(p_SH_comps, comp_weights)) / sum(comp_weights) if comp_weights else 0
        weighted_comp_prob_total = sum(p * w for p, w in zip(p_SH_comps, comp_weights)) / sum(weights)

        p_SH_noncomps = [functions.calculate_double_mutant_probablity(seq, p1_reduced, p2_reduced, J, min_pos, max_pos) for seq in noncomp_seqs]
        average_p_non_comp_subcategory = sum(p_SH_noncomps) / len(p_SH_noncomps) if p_SH_noncomps else 0
        average_p_noncomp_total = sum(p_SH_noncomps) / len_all_seqs if len_all_seqs else 0
        weighted_noncomp_prob_subcategory = sum(p * w for p, w in zip(p_SH_noncomps, non_comp_weights)) / sum(non_comp_weights) if non_comp_weights else 0
        weighted_noncomp_prob_total = sum(p * w for p, w in zip(p_SH_noncomps, non_comp_weights)) / sum(weights)


        # Calculate the actual_p (observed f)values ########################################
        flip_actual_p_total = flip_with_DMC_count / len_all_seqs
        flip_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(flip_seqs, flip_weights))
        flip_actual_p_weighted_total = flip_with_DMC_weights_sum / sum(weights) if flip_seqs else 0
        flip_actual_p_subcategory = flip_with_DMC_count / flip_counts if flip_counts > 0 else 0
        flip_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(flip_seqs, flip_weights)) / sum(flip_weights) if flip_seqs else 0

        non_flip_actual_p_total = non_flip_with_DMC_count / len_all_seqs
        non_flip_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(non_flip_seqs, non_flip_weights))
        non_flip_actual_p_weighted_total = non_flip_with_DMC_weights_sum / sum(weights) if non_flip_seqs else 0
        non_flip_actual_p_subcategory = non_flip_with_DMC_count / non_flip_counts if non_flip_counts > 0 else 0
        non_flip_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(non_flip_seqs, non_flip_weights)) / sum(non_flip_weights) if non_flip_seqs else 0

        ##############
        gof_actual_p_total = gof_with_DMC_count / len_all_seqs
        gof_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(gof_seqs, gof_weights))
        gof_actual_p_weighted_total = gof_with_DMC_weights_sum / sum(weights) if gof_seqs else 0
        gof_actual_p_subcategory = gof_with_DMC_count / gof_counts if gof_counts > 0 else 0
        gof_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(gof_seqs, gof_weights)) / sum(gof_weights) if gof_seqs else 0

        rescue_actual_p_total = rescue_with_DMC_count / len_all_seqs
        rescue_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(rescue_seqs, rescue_weights))
        rescue_actual_p_weighted_total = rescue_with_DMC_weights_sum / sum(weights) if rescue_seqs else 0
        rescue_actual_p_subcategory = rescue_with_DMC_count / rescue_count if rescue_count > 0 else 0
        rescue_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(rescue_seqs, rescue_weights)) / sum(rescue_weights) if rescue_seqs else 0


        comp_actual_p_total = compensatory_with_DMC_count / len_all_seqs
        comp_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(comp_seqs, comp_weights))
        comp_actual_p_weighted_total = comp_with_DMC_weights_sum / sum(weights) if comp_seqs else 0
        comp_actual_p_subcategory = compensatory_with_DMC_count / compensatory_counts if compensatory_counts > 0 else 0
        comp_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(comp_seqs, comp_weights)) / sum(comp_weights) if comp_seqs else 0

        noncomp_actual_p_total = noncompensatory_with_DMC_count / len_all_seqs
        noncomp_with_DMC_weights_sum = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(noncomp_seqs, non_comp_weights))
        noncomp_actual_p_weighted_total = noncomp_with_DMC_weights_sum / sum(weights) if noncomp_seqs else 0
        noncomp_actual_p_subcategory = noncompensatory_with_DMC_count / noncompensatory_counts if noncompensatory_counts > 0 else 0
        noncomp_actual_p_weighted_subcategory = sum((1 if seq[pos1 - min_pos] == mt1 and seq[pos2 - min_pos] == mt2 else 0) * w for seq, w in zip(noncomp_seqs, non_comp_weights)) / sum(non_comp_weights) if noncomp_seqs else 0

        # Append data for each epistasis subset
        #non-weighted
        csv_data.append(['total'])
        csv_data.append([pair, 'Gain_of_function', gof_counts,gof_with_DMC_count, average_p_gof_total, gof_actual_p_total, average_p_gof_subcategory, gof_actual_p_subcategory])
        csv_data.append([pair, 'rescue', rescue_count,rescue_with_DMC_count, average_p_rescue_total, rescue_actual_p_total, average_p_rescue_subcategory, rescue_actual_p_subcategory])
        csv_data.append([pair, 'compensatory', compensatory_counts,compensatory_with_DMC_count, average_p_compensatory_total, comp_actual_p_total, average_p_compensatory_subcategory, comp_actual_p_subcategory])
        csv_data.append([pair, 'non_compensatory', noncompensatory_counts, noncompensatory_with_DMC_count, average_p_noncomp_total, noncomp_actual_p_total, average_p_non_comp_subcategory, noncomp_actual_p_subcategory])
        csv_data.append([])
        # csv_data.append(['total'])
        csv_data.append([pair, 'flipped', flip_counts, flip_with_DMC_count, average_p_flip_total, flip_actual_p_total, average_p_flip_subcategory, flip_actual_p_subcategory])
        csv_data.append([pair, 'non_flipped', non_flip_counts, non_flip_with_DMC_count, average_p_nonflip_total, non_flip_actual_p_total, average_p_nonflip_subcategory, non_flip_actual_p_subcategory])
        csv_data.append([])
        #weighted
        csv_weighted_data.append(['total'])
        csv_weighted_data.append([pair,'Gain_of_function', sum(gof_weights),gof_with_DMC_weights_sum, weighted_gof_prob_total, gof_actual_p_weighted_total, weighted_gof_prob_subcategory, gof_actual_p_weighted_subcategory])
        csv_weighted_data.append([pair,'rescue', sum(rescue_weights), rescue_with_DMC_weights_sum, weighted_rescue_prob_total, rescue_actual_p_weighted_total, weighted_rescue_prob_subcategory, rescue_actual_p_weighted_subcategory])
        csv_weighted_data.append([pair,'compensatory', sum(comp_weights), comp_with_DMC_weights_sum, weighted_comp_prob_total, comp_actual_p_weighted_total, weighted_comp_prob_subcategory, comp_actual_p_weighted_subcategory])
        csv_weighted_data.append([pair,'non_compensatory', sum(non_comp_weights), noncomp_with_DMC_weights_sum, weighted_noncomp_prob_total, noncomp_actual_p_weighted_total, weighted_noncomp_prob_subcategory, noncomp_actual_p_weighted_subcategory])
        csv_weighted_data.append([])
        # csv_weighted_data.append(['total'])
        csv_weighted_data.append([pair,'flipped', sum(flip_weights), flip_with_DMC_weights_sum, weighted_flip_prob_total, flip_actual_p_weighted_total, weighted_flip_prob_subcategory, flip_actual_p_weighted_subcategory])
        csv_weighted_data.append([pair,'non_flipped', sum(non_flip_weights), non_flip_with_DMC_weights_sum, weighted_nonflip_prob_total, non_flip_actual_p_weighted_total, weighted_nonflip_prob_subcategory, weighted_nonflip_prob_subcategory])
        csv_weighted_data.append([])

        with open(output_csv, 'w', newline='') as csvfile:
            csv_writer = csv.writer(csvfile)
            csv_writer.writerow(['mutation_pair', 'epistasis_subset', 'num_seqs', 'num_with_DMC','average_p_total', 'observed_f_total','average_p_subcategory', 'observed_f_subcategory','', 'weights_sum','weights_sum_with_DMC','weighted_average_p_total', 'weighted_observed_f_total', 'weighted_average_p_subcategory', 'weighted_observed_f_subcategory'])
            count = 0 
            for row, weighted_row in zip(csv_data, csv_weighted_data):
                if row == [] or weighted_row == []:
                    csv_writer.writerow([])
                elif row == ['total'] or weighted_row == ['total']:
                    #csv_total_data.append([IN_pair,'Total', total_counts, average_p_all, actual_p_all, weighted_all_prob, actual_p_all_weighted])
                    csv_writer.writerow([f"{csv_total_data[count][0]}", f"{csv_total_data[count][1]}", f"{csv_total_data[count][2]}", f"{csv_total_data[count][3]:}", f"{csv_total_data[count][4]:.4f}", f"{csv_total_data[count][5]:.4f}", f"{csv_total_data[count][6]:.4f}",f"{csv_total_data[count][7]:.4f}", '', f"{csv_total_data[count][8]:.4f}", f"{csv_total_data[count][9]:.4f}", f"{csv_total_data[count][10]:.4f}", f"{csv_total_data[count][11]:.4f}", f"{csv_total_data[count][12]:.4f}", f"{csv_total_data[count][13]:.4f}"])
                    count += 1
                else:
                    csv_writer.writerow([f"{row[0]}", f"{row[1]}", f"{row[2]}", f"{row[3]}", f"{row[4]:.4f}", f"{row[5]:.4f}", f"{row[6]:.4f}", f"{row[7]:.4f}", '', f"{weighted_row[2]:.4f}", f"{weighted_row[3]:.4f}", f"{weighted_row[4]:.4f}", f"{weighted_row[5]:.4f}", f"{weighted_row[6]:.4f}", f"{weighted_row[7]:.4f}"])
            # Append the total data for each mutation pair
    print(f"CSV file {output_csv} created successfully.")

In [100]:
# PR_weights_path = 'PR/data/pr.exper.weights.txt'
# len_PR_all_seqs = len(PR_all_seq)
# with open(PR_weights_path, 'r') as f:
#     PR_weights = [float(line.strip()) for line in f]

# # Ensure the weights list matches the IN_all_seq list
# assert len(PR_weights) == len_PR_all_seqs, "Weights and sequences must have the same length."

# PR_pairs = [
#     'D30N-N88D',
# ]
# output_probs('PR', 1,99,PR_all_seq, PR_consensus_seq, PR_redux, PR_pairs, PR_weights_path, PR_J, 'D30N-N88D_probabilities_test.csv')

In [101]:
PR_seq_indexes = [219,222,236]
print('my previous calculation')
for idx in PR_seq_indexes:
    seq = PR_all_seq[idx]
    de1_p, de2_p, de12_p, dde_d = functions.calculate_dde_v2(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    _,_,_, dde_j = functions.calculate_dde_v3(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    residue_at_30 = PR_all_seq_unreduced[idx][30-1]
    residue_at_88 = PR_all_seq_unreduced[idx][88-1]
    print(f"Index: {idx}, sequence{residue_at_30, residue_at_88},D30N dE: {de1_p:.4f}, N88D dE: {de2_p:.4f},   dE double: {de12_p:.4f}, ddE from double: {dde_d:.4f},ddE from J: {dde_j:.4f}")

print('new caculation')
for idx in PR_seq_indexes:
    seq = PR_all_seq[idx]
    _,_,de12_actual,_ = functions.calculate_dde_v2(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    de1_p, de2_p, _, dde_j = functions.calculate_dde_v3(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    dde_d = de12_actual - de1_p - de2_p

    residue_at_30 = PR_all_seq_unreduced[idx][30-1]
    residue_at_88 = PR_all_seq_unreduced[idx][88-1]
    print(f"Index: {idx}, sequence: {residue_at_30,residue_at_88}, D30N dE: {de1_p:.4f}, N88D dE: {de2_p:.4f},   dE double: {de12_actual:.4f}, ddE from double: {dde_d:.4f}, ddE from J: {dde_j:.4f}")


my previous calculation
Index: 219, sequence('N', 'N'),D30N dE: -1.6287, N88D dE: -4.7510,   dE double: -1.7689, ddE from double: 4.6109,ddE from J: 4.6109
Index: 222, sequence('N', 'D'),D30N dE: 0.0309, N88D dE: -3.6556,   dE double: 0.9862, ddE from double: 4.6109,ddE from J: 4.6109
Index: 236, sequence('D', 'S'),D30N dE: -2.3684, N88D dE: -3.9242,   dE double: -1.6818, ddE from double: 4.6109,ddE from J: 4.6109
new caculation
Index: 219, sequence: ('N', 'N'), D30N dE: -1.6287, N88D dE: -0.1402,   dE double: -1.7689, ddE from double: -0.0000, ddE from J: 4.6109
Index: 222, sequence: ('N', 'D'), D30N dE: 4.6418, N88D dE: 0.9553,   dE double: 0.9862, ddE from double: -4.6109, ddE from J: 4.6109
Index: 236, sequence: ('D', 'S'), D30N dE: -2.5030, N88D dE: -3.9242,   dE double: -1.6818, ddE from double: 4.7454, ddE from J: 4.6109


In [102]:
PR_seq_indexes = [219,222,236]
print('from J, with substitution')
for idx in PR_seq_indexes:
    seq = PR_all_seq[idx]
    de1, de2, de12, dde = functions.calculate_dde_from_J_with_substitution(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    residue_at_30 = PR_all_seq_unreduced[idx][30-1]
    residue_at_88 = PR_all_seq_unreduced[idx][88-1]
    print(f"Index: {idx}, sequence{residue_at_30, residue_at_88},D30N dE: {de1:.4f}, N88D dE: {de2:.4f},   dE double: {de12:.4f}, ddE: {dde:.4f}")

print('from J, without substitution')
for idx in PR_seq_indexes:
    seq = PR_all_seq[idx]
    de1, de2, de12, dde = functions.calculate_dde_from_J_no_substitution(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    residue_at_30 = PR_all_seq_unreduced[idx][30-1]
    residue_at_88 = PR_all_seq_unreduced[idx][88-1]
    print(f"Index: {idx}, sequence{residue_at_30, residue_at_88},D30N dE: {de1:.4f}, N88D dE: {de2:.4f},   dE double: {de12:.4f}, ddE: {dde:.4f}")

print('from dE, with substitution')
for idx in PR_seq_indexes:
    seq = PR_all_seq[idx]
    de1, de2, de12, dde = functions.calculate_dde_from_de_with_substitution(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    residue_at_30 = PR_all_seq_unreduced[idx][30-1]
    residue_at_88 = PR_all_seq_unreduced[idx][88-1]
    print(f"Index: {idx}, sequence{residue_at_30, residue_at_88},D30N dE: {de1:.4f}, N88D dE: {de2:.4f},   dE double: {de12:.4f}, ddE: {dde:.4f}")

print('from dE, without substitution')
for idx in PR_seq_indexes:
    seq = PR_all_seq[idx]
    de1, de2, de12, dde = functions.calculate_dde_from_de_no_substitution(reduced_pair1, reduced_pair2, seq, PR_J, 1, 99)
    residue_at_30 = PR_all_seq_unreduced[idx][30-1]
    residue_at_88 = PR_all_seq_unreduced[idx][88-1]
    print(f"Index: {idx}, sequence{residue_at_30, residue_at_88},D30N dE: {de1:.4f}, N88D dE: {de2:.4f},   dE double: {de12:.4f}, ddE: {dde:.4f}")

from J, with substitution
Index: 219, sequence('N', 'N'),D30N dE: -1.6287, N88D dE: -0.1402,   dE double: 2.8420, ddE: 4.6109
Index: 222, sequence('N', 'D'),D30N dE: 4.6418, N88D dE: 0.9553,   dE double: 10.2079, ddE: 4.6109
Index: 236, sequence('D', 'S'),D30N dE: -2.5030, N88D dE: -3.9242,   dE double: -1.8163, ddE: 4.6109
from J, without substitution
Index: 219, sequence('N', 'N'),D30N dE: -1.6287, N88D dE: -0.1402,   dE double: -1.7689, ddE: 0.0000
Index: 222, sequence('N', 'D'),D30N dE: 4.6418, N88D dE: 0.9553,   dE double: 5.5971, ddE: 0.0000
Index: 236, sequence('D', 'S'),D30N dE: -2.5030, N88D dE: -3.9242,   dE double: -1.6818, ddE: 4.7454
from dE, with substitution
Index: 219, sequence('N', 'N'),D30N dE: -1.6287, N88D dE: -4.7510,   dE double: -1.7689, ddE: 4.6109
Index: 222, sequence('N', 'D'),D30N dE: 0.0309, N88D dE: -3.6556,   dE double: 0.9862, ddE: 4.6109
Index: 236, sequence('D', 'S'),D30N dE: -2.3684, N88D dE: -3.9242,   dE double: -1.6818, ddE: 4.6109
from dE, without 

In [103]:
PR_seq_indexes = [219,222,236]
PR_pairs = ['D30N-N88D']
pair = PR_pairs[0]
pair1, pair2 = functions.split_pairs(pair)
reduced_pair1 = functions.unreduced_to_reduced(PR_redux, pair1)
reduced_pair2 = functions.unreduced_to_reduced(PR_redux, pair2)
wt1, pos1, mt1 = functions.split_pair(reduced_pair1)
wt2, pos2, mt2 = functions.split_pair(reduced_pair2)

print('sequence with WT at both positions 30 and 88')

for idx in PR_seq_indexes:
    seq = PR_all_seq[idx]

    seq_list = list(seq)
    
    seq_list[pos1 - 1] = wt1
    seq_list[pos2 - 1] = wt2
    seq_wt = ''.join(seq_list)

    # print(f"index={idx}:, sequence={seq_wt[pos30 - 1], seq_wt[pos88 - 1]}")
    # print(seq_wt)
    #EM1
    mut30_reduced = functions.unreduced_to_reduced(PR_redux, "D30N")
    wt30, pos30, mt30 = functions.split_pair(mut30_reduced)
    seq_list = list(seq_wt)
    seq_list[pos30 - 1] = mt30   # mutate position 30 to N (in reduced alphabet)
    seqM1 = "".join(seq_list)

    EM1 = functions.calculate_e(seqM1, PR_J, 1, 99)
    
    #EM2
    mut88_reduced = functions.unreduced_to_reduced(PR_redux, "N88D")
    wt88, pos88, mt88 = functions.split_pair(mut88_reduced)
    seq_list = list(seq_wt)
    seq_list[pos88 - 1] = mt88   # mutate position 88 to D (in reduced alphabet)
    seqM2 = "".join(seq_list)

    EM2 = functions.calculate_e(seqM2, PR_J, 1, 99)

    #EM1M2
    seq_list = list(seq_wt)
    seq_list[pos30 - 1] = mt30   # mutate position 30 to N (in reduced alphabet)
    seq_list[pos88 - 1] = mt88   # mutate position 88 to D (in reduced alphabet)
    seqM1M2 = "".join(seq_list)

    EM1M2 = functions.calculate_e(seqM1M2, PR_J, 1, 99)

    #Eseq
    Eseq = functions.calculate_e(seq_wt, PR_J, 1, 99)

    #EW1W2
    seq_list = list(seq_wt)
    seq_list[pos30 - 1] = wt30   # mutate position 30 back to D (in reduced alphabet)
    seq_list[pos88 - 1] = wt88   # mutate position 88 back to N (in reduced alphabet)
    seqW1W2 = "".join(seq_list)
    EW1W2 = functions.calculate_e(seqW1W2, PR_J, 1, 99)

    #sequence residues at positions 30 and 88 in the unreduced sequence
    residue_at_30 = PR_all_seq_unreduced[idx][30-1]
    residue_at_88 = PR_all_seq_unreduced[idx][88-1]
    #EConsensus
    EConsensus = functions.calculate_e(PR_consensus_seq, PR_J, 1, 99)
    print(f"index={idx}:, sequence={residue_at_30,residue_at_88}, EM1={EM1}, EM2={EM2}, EM1M2={EM1M2}, Eseq={Eseq}, EW1W2={EW1W2}, EConsensus={EConsensus}")

#################################################
print('sequence with actual residues at positions 30 and 88')
for idx in PR_seq_indexes:
    seq = PR_all_seq[idx]
    # print(f"index={idx}:, sequence={seq[pos30 - 1], seq[pos88 - 1]}")
    #EM1
    mut30_reduced = functions.unreduced_to_reduced(PR_redux, "D30N")
    wt30, pos30, mt30 = functions.split_pair(mut30_reduced)
    seq_list = list(seq)
    seq_list[pos30 - 1] = mt30   # mutate position 30 to N (in reduced alphabet)
    seqM1 = "".join(seq_list)

    EM1 = functions.calculate_e(seqM1, PR_J, 1, 99)
    
    #EM2
    mut88_reduced = functions.unreduced_to_reduced(PR_redux, "N88D")
    wt88, pos88, mt88 = functions.split_pair(mut88_reduced)
    seq_list = list(seq)
    seq_list[pos88 - 1] = mt88   # mutate position 88 to D (in reduced alphabet)
    seqM2 = "".join(seq_list)

    EM2 = functions.calculate_e(seqM2, PR_J, 1, 99)

    #EM1M2
    seq_list = list(seq)
    seq_list[pos30 - 1] = mt30   # mutate position 30 to N (in reduced alphabet)
    seq_list[pos88 - 1] = mt88   # mutate position 88 to D (in reduced alphabet)
    seqM1M2 = "".join(seq_list)

    EM1M2 = functions.calculate_e(seqM1M2, PR_J, 1, 99)

    #Eseq
    Eseq = functions.calculate_e(seq, PR_J, 1, 99)

    #EW1W2
    seq_list = list(seq)
    seq_list[pos30 - 1] = wt30   # mutate position 30 back to D (in reduced alphabet)
    seq_list[pos88 - 1] = wt88   # mutate position 88 back to N (in reduced alphabet)
    seqW1W2 = "".join(seq_list)
    EW1W2 = functions.calculate_e(seqW1W2, PR_J, 1, 99)

    #sequence residues at positions 30 and 88 in the unreduced sequence
    residue_at_30 = PR_all_seq_unreduced[idx][30-1]
    residue_at_88 = PR_all_seq_unreduced[idx][88-1]
    #EConsensus
    EConsensus = functions.calculate_e(PR_consensus_seq, PR_J, 1, 99)
    print(f"index={idx}:, sequence={residue_at_30,residue_at_88}, EM1={EM1}, EM2={EM2}, EM1M2={EM1M2}, Eseq={Eseq}, EW1W2={EW1W2}, EConsensus={EConsensus}")

sequence with WT at both positions 30 and 88
index=219:, sequence=('N', 'N'), EM1=-562.591796875, EM2=-559.4698486328125, EM1M2=-562.4517822265625, Eseq=-564.220703125, EW1W2=-564.220703125, EConsensus=-577.5924682617188
index=222:, sequence=('N', 'D'), EM1=-561.417236328125, EM2=-557.73095703125, EM1M2=-562.3726196289062, Eseq=-561.3865356445312, EW1W2=-561.3865356445312, EConsensus=-577.5924682617188
index=236:, sequence=('D', 'S'), EM1=-550.84130859375, EM2=-549.2858276367188, EM1M2=-551.528076171875, Eseq=-553.2100219726562, EW1W2=-553.2100219726562, EConsensus=-577.5924682617188
sequence with actual residues at positions 30 and 88
index=219:, sequence=('N', 'N'), EM1=-562.591796875, EM2=-562.4517822265625, EM1M2=-562.4517822265625, Eseq=-562.591796875, EW1W2=-564.220703125, EConsensus=-577.5924682617188
index=222:, sequence=('N', 'D'), EM1=-562.3726196289062, EM2=-562.3726196289062, EM1M2=-562.3726196289062, Eseq=-562.3726196289062, EW1W2=-561.3865356445312, EConsensus=-577.592468